# kc-dataset


In [7]:
import os
import sys
import torch
import torchvision.transforms as T
from torch.utils.data import DataLoader
from torch.utils.data import ConcatDataset
import pyarrow as pa
import pyarrow.parquet as pq

# Ajouter le chemin du package au PYTHONPATH
sys.path.insert(0, '/Users/checkkoutame/Desktop/Renare/kc-dataset')

from kc_dataset import StratifiedHashSplitter, ParquetDataset, IOConfig, add_uuid_to_parquet


## Spliting

In [8]:
# con
io_cfg = IOConfig(
    id_col="sample_id",
    class_col="class_name",
    image_bytes_col="image_bytes",
    split_col="split"
)

# stratified splitting strategy
splitter = StratifiedHashSplitter(
    by_cols=("class_name",),  # on peut changer les colonnes par lesquelles on veut faire le split, prendre par exemple le sample ID
    ratios=(0.8, 0.1, 0.1)
)


In [9]:
# genère le fichier avec les splits
original_table = pq.read_table("data/samples.parquet")
table_with_splits = splitter.apply(original_table)
new_path = "data/samples_with_split.parquet"
pq.write_table(table_with_splits, new_path)

/var/folders/hw/s6q37kzd6l7bvch2rztj09lw0000gn/T/ipykernel_37143/4011673727.py:3: FutureWarning: promote has been superseded by promote_options='default'.
  table_with_splits = splitter.apply(original_table)


In [10]:
# Ajoute une colonne "sample_uuid" unique à chaque ligne du fichier parquet
add_uuid_to_parquet(
    input_path="data/samples_with_split.parquet",
    output_path="data/samples_with_split.parquet",  # Écrase le fichier existant
    uuid_column="sample_uuid",
    overwrite_existing=False  # Si la colonne existe déjà, lève une erreur
)

print("UUIDs ajoutés avec succès!")


UUIDs ajoutés avec succès!


# Import dataset

In [11]:
parquet_path = os.path.join(os.getcwd(), "data/samples_with_split.parquet")

In [6]:
# Image transform

mean = torch.tensor([0.5, 0.5, 0.5])
std = torch.tensor([0.5, 0.5, 0.5])

transform = T.Compose([T.Resize((64,64)),
                       T.ToTensor(),
                       T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
                       T.ConvertImageDtype(torch.float32)
                      ])

unnormalize = T.Normalize((-mean / std).tolist(), (1.0 / std).tolist())



train_dataset = ParquetDataset( parquet_path, splitter=None, io_cfg=io_cfg, split="train",transform=transform )
val_dataset = ParquetDataset( parquet_path, splitter=None,io_cfg=io_cfg, split="val",transform=transform )
test_dataset = ParquetDataset(parquet_path, splitter=None, io_cfg=io_cfg, split="test",transform=transform)

# Combiner val + test
validation_dataset = ConcatDataset([val_dataset, test_dataset])


train_dl = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dl = DataLoader(validation_dataset, batch_size=32, shuffle=True)

dataloaders = {'train': train_dl, 'val': val_dl}
dataset_sizes = {'train': len(train_dataset), 'val': len(validation_dataset)}
